In [ ]:
import pandas as pd
import numpy as np
import itertools
from pathlib import Path

In [ ]:
# Input
summary_zscore_file = Path("/Users/miasmacbook/Desktop/6-months_project/data/Similarity/Zscore_disease_avg_similarity_summary.csv")
summary_TPM_file = Path("/Users/miasmacbook/Desktop/6-months_project/data/Similarity/TPM_disease_avg_similarity_summary.csv")
summary_log2_file = Path("/Users/miasmacbook/Desktop/6-months_project/data/Similarity/log2_disease_avg_similarity_summary.csv")
matrix_zscore_file = Path("/Users/miasmacbook/Desktop/6-months_project/database/gene_gene_correlation_zscore_updated.csv")
matrix_TPM_file = Path("/Users/miasmacbook/Desktop/6-months_project/database/Matrix/gene_gene_correlation_TPM_updated.csv")
matrix_log2_file = Path("/Users/miasmacbook/Desktop/6-months_project/database/gene_gene_correlation_log2_updated.csv")

# output 
output_dir = Path("/Users/miasmacbook/Desktop/6-months_project/data/Similariy/random_similarity")
output_dir.mkdir(parents=True, exist_ok=True)

summary_zscore_out = output_dir / "Zscore_disease_vs_random_summary.csv"
random_zscore_out = output_dir / "Zscore_random_similarity_distribution.csv"
summary_TPM_out = output_dir / "TPM_disease_vs_random_summary.csv"
random_TPM_out = output_dir / "TPM_random_similarity_distribution.csv"
summary_log2_out = output_dir / "log2_disease_vs_random_summary.csv"
random_log2_out = output_dir / "log2_random_similarity_distribution.csv"

In [ ]:
n_random = 1000
seed = 4202

In [ ]:
def get_avg_similarity(gene_list, sim_matrix):
    vals = []

    for g1, g2 in itertools.combinations(gene_list, 2):
        val = sim_matrix.loc[g1, g2]
        if pd.notna(val):
            vals.append(val)

    if len(vals) == 0:
        return np.nan, 0

    return float(np.mean(vals)), len(vals)

# Main function for three matrics

In [ ]:
def run_random_baseline(summary_file, matrix_file, summary_out, random_out, label, n_random=1000, seed=4202):
    rng = np.random.default_rng(seed)

    print(f"=== {label} ===")
    print("Loading disease summary...")
    df = pd.read_csv(summary_file)

    print("Loading similarity matrix...")
    sim_matrix = pd.read_csv(matrix_file, sep=",", index_col=0)
    sim_matrix.index = sim_matrix.index.astype(str).str.strip()
    sim_matrix.columns = sim_matrix.columns.astype(str).str.strip()

    common_genes = sim_matrix.index.intersection(sim_matrix.columns)
    sim_matrix = sim_matrix.loc[common_genes, common_genes]
    gene_universe = sim_matrix.index.tolist()

    print(f"Summary rows: {len(df)}")
    print(f"Matrix shape: {sim_matrix.shape}")
    print(f"Genes available for random sampling: {len(gene_universe)}")

    random_rows = []
    new_summary_rows = []

    for i, row in df.iterrows():
        disease_id = row["diseaseFromSourceMappedId"]
        n_genes = int(row["n_genes_used"]) if pd.notna(row["n_genes_used"]) else 0
        observed_avg = row["avg_pairwise_similarity"]

        print(f"[{i+1}/{len(df)}] {disease_id} (n={n_genes})")

        if n_genes < 2:
            row_dict = row.to_dict()
            row_dict["random_mean_similarity"] = np.nan
            row_dict["random_sd_similarity"] = np.nan
            row_dict["random_n_iter_used"] = 0
            row_dict["empirical_p_random_ge_observed"] = np.nan
            row_dict["z_score_vs_random"] = np.nan
            new_summary_rows.append(row_dict)
            continue

        if n_genes > len(gene_universe):
            print(f"  Skipping random sampling: n_genes ({n_genes}) > gene universe ({len(gene_universe)})")
            row_dict = row.to_dict()
            row_dict["random_mean_similarity"] = np.nan
            row_dict["random_sd_similarity"] = np.nan
            row_dict["random_n_iter_used"] = 0
            row_dict["empirical_p_random_ge_observed"] = np.nan
            row_dict["z_score_vs_random"] = np.nan
            new_summary_rows.append(row_dict)
            continue

        random_avgs = []

        for j in range(n_random):
            random_genes = rng.choice(gene_universe, size=n_genes, replace=False).tolist()
            rand_avg, rand_n_pairs = get_avg_similarity(random_genes, sim_matrix)

            if pd.notna(rand_avg):
                random_avgs.append(rand_avg)
                random_rows.append({
                    "disease_id": disease_id,
                    "iteration": j + 1,
                    "n_genes": n_genes,
                    "random_avg_similarity": rand_avg
                })

        if len(random_avgs) > 0 and pd.notna(observed_avg):
            random_mean = float(np.mean(random_avgs))
            random_sd = float(np.std(random_avgs, ddof=1)) if len(random_avgs) > 1 else np.nan
            empirical_p = (np.sum(np.array(random_avgs) >= observed_avg) + 1) / (len(random_avgs) + 1)

            if pd.notna(random_sd) and random_sd != 0:
                z_score = (observed_avg - random_mean) / random_sd
            else:
                z_score = np.nan
        else:
            random_mean = np.nan
            random_sd = np.nan
            empirical_p = np.nan
            z_score = np.nan

        row_dict = row.to_dict()
        row_dict["random_mean_similarity"] = random_mean
        row_dict["random_sd_similarity"] = random_sd
        row_dict["random_n_iter_used"] = len(random_avgs)
        row_dict["empirical_p_random_ge_observed"] = empirical_p
        row_dict["z_score_vs_random"] = z_score
        new_summary_rows.append(row_dict)

    new_summary_df = pd.DataFrame(new_summary_rows)
    random_df = pd.DataFrame(random_rows)

    new_summary_df.to_csv(summary_out, index=False)
    random_df.to_csv(random_out, index=False)

    print(f"\nDone for {label}.")
    print(f"Saved summary: {summary_out}")
    print(f"Saved random distributions: {random_out}")
    print()

In [ ]:
run_random_baseline(
    summary_file=summary_zscore_file,
    matrix_file=matrix_zscore_file,
    summary_out=summary_zscore_out,
    random_out=random_zscore_out,
    label="Zscore",
    n_random=n_random,
    seed=seed
)

run_random_baseline(
    summary_file=summary_TPM_file,
    matrix_file=matrix_TPM_file,
    summary_out=summary_TPM_out,
    random_out=random_TPM_out,
    label="TPM",
    n_random=n_random,
    seed=seed
)

run_random_baseline(
    summary_file=summary_log2_file,
    matrix_file=matrix_log2_file,
    summary_out=summary_log2_out,
    random_out=random_log2_out,
    label="log2",
    n_random=n_random,
    seed=seed
)